# US Road Network Analysis
           MSDA9215: Big Data Analytics
           Group Members: 1. SHYAKA Ivan 101229
                          2. RWEGO Marie Vaillante 101225
                          3. Tumukunde Sandra 101483
           Date: June 2026

## 1. Environment Setup

In [31]:
import subprocess
subprocess.run(["pip", "install", "neo4j", "plotly", "pandas", "networkx"], check=True)

CompletedProcess(args=['pip', 'install', 'neo4j', 'plotly', 'pandas', 'networkx'], returncode=0)

## 2. Database Connection

In [2]:
import os
from neo4j import GraphDatabase
import math

NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, 
                               auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    result = session.run("RETURN 'Connected!' AS message")
    print(result.single()["message"])

Connected!


## 3. Data Loading

In [30]:
import math

def parse_usa_file(filepath):
    nodes = []
    edges = []
    with open(filepath, 'r') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]
    
    first_line = lines[0].split()
    num_nodes = int(first_line[0])
    num_edges = int(first_line[1])
    
    print(f"Expected nodes: {num_nodes}, Expected edges: {num_edges}")
    
    for i in range(1, num_nodes + 1):
        parts = lines[i].split()
        nodes.append((int(parts[0]), float(parts[1]), float(parts[2])))
    
    for i in range(num_nodes + 1, num_nodes + 1 + num_edges):
        parts = lines[i].split()
        edges.append((int(parts[0]), int(parts[1])))
    
    return nodes, edges

filepath = r"C:\Users\user\OneDrive\Desktop\Masters_Program\Big_Data_Analytics\ClassWork\Project2\US Road Network\usa.txt"
# nodes, edges = parse_usa_file(filepath)

# print(f"Nodes loaded: {len(nodes)}")
# print(f"Edges loaded: {len(edges)}")
# print(f"Sample node: {nodes[0]}")
# print(f"Sample edge: {edges[0]}")

print("Note: Data already parsed and loaded into Neo4j.")

Note: Data already parsed and loaded into Neo4j.


In [29]:
# Creating uniqueness constraint for faster loading
with driver.session() as session:
    session.run("""
        CREATE CONSTRAINT intersection_id IF NOT EXISTS 
        FOR (n:Intersection) REQUIRE n.id IS UNIQUE
    """)
    print("Constraint created!")

# Loading nodes in batches of 1000
def load_nodes_batch(tx, batch):
    tx.run("""
        UNWIND $batch AS row
        CREATE (:Intersection {id: row.id, x: row.x, y: row.y})
    """, batch=batch)

def load_nodes(driver, nodes, batch_size=1000):
    with driver.session() as session:
        batch = []
        for i, (node_id, x, y) in enumerate(nodes):
            batch.append({"id": node_id, "x": x, "y": y})
            if len(batch) == batch_size:
                session.execute_write(load_nodes_batch, batch)
                batch = []
                if i % 10000 == 0:
                    print(f"Loaded {i} nodes...")
        if batch:
            session.execute_write(load_nodes_batch, batch)
    print(f"Done! All {len(nodes)} nodes loaded.")

#  load_nodes(driver, nodes)

Constraint created!


In [6]:
# Building coordinate lookup for distance calculation
node_coords = {node_id: (x, y) for node_id, x, y in nodes}

def load_edges_batch(tx, batch):
    tx.run("""
        UNWIND $batch AS row
        MATCH (a:Intersection {id: row.from_id})
        MATCH (b:Intersection {id: row.to_id})
        MERGE (a)-[:ROAD {distance: row.distance}]->(b)
        MERGE (b)-[:ROAD {distance: row.distance}]->(a)
    """, batch=batch)

def load_edges(driver, edges, node_coords, batch_size=500):
    with driver.session() as session:
        batch = []
        for i, (from_id, to_id) in enumerate(edges):
            x1, y1 = node_coords[from_id]
            x2, y2 = node_coords[to_id]
            dist = math.sqrt((x2-x1)**2 + (y2-y1)**2)
            batch.append({
                "from_id": from_id,
                "to_id": to_id,
                "distance": round(dist, 2)
            })
            if len(batch) == batch_size:
                session.execute_write(load_edges_batch, batch)
                batch = []
                if i % 10000 == 0:
                    print(f"Loaded {i} edges...")
        if batch:
            session.execute_write(load_edges_batch, batch)
    print(f"Done! All {len(edges)} edges loaded.")

# load_edges(driver, edges, node_coords)

Done! All 121961 edges loaded.


## 4. NetworkX Graph Construction

In [28]:
import networkx as nx

# Pulling all edges from Neo4j and build NetworkX graph
print("Building NetworkX graph...")

G = nx.Graph() 

with driver.session() as session:
    # Adding all nodes to networkx graph
    result = session.run("""
        MATCH (n:Intersection) 
        RETURN n.id AS id, n.x AS x, n.y AS y
    """)
    for record in result:
        G.add_node(record["id"], x=record["x"], y=record["y"])
    
    # Adding all edges to NetworkX graph    
    result = session.run("""
        MATCH (a:Intersection)-[r:ROAD]->(b:Intersection)
        WHERE a.id < b.id
        RETURN a.id AS from_id, b.id AS to_id, r.distance AS distance
    """)
    for record in result:
        G.add_edge(record["from_id"], record["to_id"], 
                   weight=record["distance"])

print(f"Graph built!")
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")

Building NetworkX graph...
Graph built!
Nodes: 87,575
Edges: 121,491


## Task 1: Total Intersections and Roads

In [18]:
# TASK 1: Total intersections and roads
with driver.session() as session:
    nodes_count = session.run("MATCH (n:Intersection) RETURN count(n) AS total").single()["total"]
    roads_count = session.run("MATCH ()-[r:ROAD]->() RETURN count(r) AS total").single()["total"]
    
    print("=" * 40)
    print(f"TASK 1: Network Overview")
    print("=" * 40)
    print(f"Total Intersections: {nodes_count:,}")
    print(f"Total Roads:         {roads_count:,}")

TASK 1: Network Overview
Total Intersections: 87,575
Total Roads:         242,982


## Task 2: Shortest Path

In [19]:
# TASK 2: Shortest path between two intersections
import networkx as nx

source = 0      # starting intersection
target = 50000  # destination intersection

print("=" * 40)
print(f"TASK 2: Shortest Path")
print(f"From Intersection {source} to {target}")
print("=" * 40)

try:
    # Finding shortest path using Dijkstra
    path = nx.dijkstra_path(G, source=source, target=target, weight='weight')
    distance = nx.dijkstra_path_length(G, source=source, target=target, weight='weight')
    
    print(f"Path found!")
    print(f"Total Distance:    {distance:,.2f} units")
    print(f"Number of roads:   {len(path)-1}")
    print(f"Number of stops:   {len(path)}")
    print(f"First 5 nodes:     {path[:5]}")
    print(f"Last 5 nodes:      {path[-5:]}")

except nx.NetworkXNoPath:
    print("No path exists between these nodes")
except nx.NodeNotFound:
    print("One of the nodes doesn't exist")

TASK 2: Shortest Path
From Intersection 0 to 50000
Path found!
Total Distance:    2,555.25 units
Number of roads:   319
Number of stops:   320
First 5 nodes:     [0, 18, 4, 1, 71743]
Last 5 nodes:      [49892, 49854, 49919, 49983, 50000]


## Task 3: High Degree Intersections

In [20]:
# TASK 3: Finding all intersections with degree greater than 3
print("=" * 40)
print("TASK 3: High Degree Intersections")
print("Intersections with degree > 3")
print("=" * 40)

# Getting degree of every node from NetworkX
high_degree_nodes = [(node, degree) for node, degree in G.degree() 
                     if degree > 3]

# Sorting by degree descending
high_degree_nodes.sort(key=lambda x: x[1], reverse=True)

print(f"Total intersections with degree > 3: {len(high_degree_nodes):,}")
print(f"\nTop 10 highest degree intersections:")
print(f"{'Intersection ID':<20} {'Degree':<10}")
print("-" * 30)
for node, degree in high_degree_nodes[:10]:
    print(f"{node:<20} {degree:<10}")

TASK 3: High Degree Intersections
Intersections with degree > 3
Total intersections with degree > 3: 19,914

Top 10 highest degree intersections:
Intersection ID      Degree    
------------------------------
2073                 6         
2581                 6         
5831                 6         
6017                 6         
9631                 6         
11356                6         
11968                6         
17391                6         
17714                6         
25259                6         


## Task 4: Betweenness Centrality

In [14]:
# TASK 4: Betweenness Centrality
print("=" * 40)
print("TASK 4: Betweenness Centrality")
print("=" * 40)
print("Calculating... this may take a few minutes...")

# Use k parameter to sample nodes for faster computation
# k=500 means sample 500 nodes instead of all 87,575
# gives a good approximation without taking hours
betweenness = nx.betweenness_centrality(G, 
                                         k=500, 
                                         weight='weight',
                                         normalized=True)

# Sort by centrality score descending
sorted_betweenness = sorted(betweenness.items(), 
                            key=lambda x: x[1], 
                            reverse=True)

print(f"\nTop 10 Most Critical Intersections:")
print(f"{'Intersection ID':<20} {'Centrality Score':<20}")
print("-" * 40)
for node, score in sorted_betweenness[:10]:
    print(f"{node:<20} {score:<20.6f}")

print(f"\nBottom 5 Least Critical Intersections:")
print(f"{'Intersection ID':<20} {'Centrality Score':<20}")
print("-" * 40)
for node, score in sorted_betweenness[-5:]:
    print(f"{node:<20} {score:<20.6f}")

TASK 4: Betweenness Centrality
Calculating... this may take a few minutes...

Top 10 Most Critical Intersections:
Intersection ID      Centrality Score    
----------------------------------------
41417                0.020152            
41511                0.017347            
87275                0.015604            
48218                0.015544            
23185                0.015053            
3248                 0.014868            
3250                 0.014852            
3256                 0.014541            
42475                0.013616            
2669                 0.013543            

Bottom 5 Least Critical Intersections:
Intersection ID      Centrality Score    
----------------------------------------
87535                0.000000            
87571                0.000000            
87572                0.000000            
87573                0.000000            
87574                0.000000            


## Tasks 5-9: Dashboard and Visualizations

In [21]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
from collections import Counter

## 1. Preparing the Data 

In [23]:
from collections import Counter

# Getting degree of every node 
degree_data = [(node, degree) for node, degree in G.degree()]
degrees = [d for _, d in degree_data]
degree_counts = Counter(degrees)

print("Degree counts (degree: number of intersections):")
for degree, count in sorted(degree_counts.items()):
    print(f"  Degree {degree}: {count:,} intersections")

# Sorting by degree 
sorted_degrees = sorted(degree_counts.items())
degree_values = [d for d, _ in sorted_degrees]
node_counts = [c for _, c in sorted_degrees]

print(f"\nDegree values available: {degree_values}")
print(f"Corresponding node counts: {node_counts}")

# Top 10 most connected 
top10 = sorted(degree_data, key=lambda x: x[1], reverse=True)[:10]
top10_ids = [str(n) for n, _ in top10]
top10_degrees = [d for _, d in top10]

print(f"\nTop 10 Most Connected Intersections:")
print(f"{'Intersection ID':<20} {'Degree':<10}")
print("-" * 30)
for node_id, degree in zip(top10_ids, top10_degrees):
    print(f"{node_id:<20} {degree:<10}")

# Categories 
low    = sum(1 for d in degrees if d <= 2)
medium = sum(1 for d in degrees if d in [3, 4])
high   = sum(1 for d in degrees if d >= 5)

print(f"\nIntersection Categories:")
print(f"{'Category':<25} {'Count':>10} {'Percentage':>12}")
print("-" * 50)
print(f"{'Low  (1-2 roads)':<25} {low:>10,} {low/len(degrees)*100:>11.1f}%")
print(f"{'Medium (3-4 roads)':<25} {medium:>10,} {medium/len(degrees)*100:>11.1f}%")
print(f"{'High (5+ roads)':<25} {high:>10,} {high/len(degrees)*100:>11.1f}%")
print("-" * 50)
print(f"{'Total':<25} {len(degrees):>10,} {'100.0%':>12}")

Degree counts (degree: number of intersections):
  Degree 1: 2,234 intersections
  Degree 2: 35,486 intersections
  Degree 3: 29,941 intersections
  Degree 4: 19,652 intersections
  Degree 5: 227 intersections
  Degree 6: 35 intersections

Degree values available: [1, 2, 3, 4, 5, 6]
Corresponding node counts: [2234, 35486, 29941, 19652, 227, 35]

Top 10 Most Connected Intersections:
Intersection ID      Degree    
------------------------------
2073                 6         
2581                 6         
5831                 6         
6017                 6         
9631                 6         
11356                6         
11968                6         
17391                6         
17714                6         
25259                6         

Intersection Categories:
Category                       Count   Percentage
--------------------------------------------------
Low  (1-2 roads)              37,720        43.1%
Medium (3-4 roads)            49,593        56.6%
High

## Task 5: Dashboard with Key Metrics

In [24]:
# Task 5: Key Metrics Dashboard

print("=" * 40)
print("TASK 5: Key Metrics Dashboard")
print("=" * 40)

fig5 = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "indicator"}, 
            {"type": "indicator"}, 
            {"type": "indicator"}]]
)

fig5.add_trace(go.Indicator(
    mode="number",
    value=G.number_of_nodes(),
    title={"text": "Total Intersections"},
    number={"font": {"size": 40, "color": "blue"}}
), row=1, col=1)

fig5.add_trace(go.Indicator(
    mode="number",
    value=G.number_of_edges(),
    title={"text": "Total Roads"},
    number={"font": {"size": 40, "color": "green"}}
), row=1, col=2)

fig5.add_trace(go.Indicator(
    mode="number",
    value=round(sum(degrees)/len(degrees), 2),
    title={"text": "Average Degree"},
    number={"font": {"size": 40, "color": "red"}}
), row=1, col=3)

fig5.update_layout(
    title_text="US Road Network - Key Metrics Dashboard",
    height=300
)
fig5.show()

TASK 5: Key Metrics Dashboard


## Task 6 & 9: Degree Distribution

In [25]:
# TASK 6 & 9: Degree Distribution Bar Chart
print("=" * 40)
print("TASK 6 & 9: Degree Distribution")
print("=" * 40)

fig6 = px.bar(
    x=degree_values,
    y=node_counts,
    labels={"x": "Degree (Number of Roads)", 
            "y": "Number of Intersections"},
    title="Degree Distribution of US Road Network",
    color=node_counts,
    color_continuous_scale="Blues"
)
fig6.update_layout(
    xaxis_title="Degree",
    yaxis_title="Number of Intersections",
    showlegend=False
)
fig6.show()

TASK 6 & 9: Degree Distribution


## Task 7: Top 10 Most Connected

In [26]:
# TASK 7: Top 10 Most Connected Intersections 
print("=" * 40)
print("TASK 7: Top 10 Most Connected")
print("=" * 40)

fig7 = px.bar(
    x=top10_ids,
    y=top10_degrees,
    labels={"x": "Intersection ID", 
            "y": "Degree"},
    title="Top 10 Most Connected Intersections",
    color=top10_degrees,
    color_continuous_scale="Reds"
)
fig7.update_layout(
    xaxis_title="Intersection ID",
    yaxis_title="Number of Connections (Degree)"
)
fig7.show()

TASK 7: Top 10 Most Connected


## Task 8: Intersection Categories

In [27]:
# TASK 8: Intersection Categories by Degree
print("=" * 40)
print("TASK 8: Intersection Categories")
print("=" * 40)

fig8 = px.pie(
    names=["Low (1-2 roads)", 
           "Medium (3-4 roads)", 
           "High (5+ roads)"],
    values=[low, medium, high],
    title="Intersection Categories by Connectivity Level",
    color_discrete_sequence=["#3498db", "#2ecc71", "#e74c3c"]
)
fig8.update_traces(textposition='inside', textinfo='percent+label')
fig8.show()

# Print summary
print(f"\nCategory Summary:")
print(f"Low  (1-2 roads): {low:,}  ({low/len(degrees)*100:.1f}%)")
print(f"Medium (3-4 roads): {medium:,}  ({medium/len(degrees)*100:.1f}%)")
print(f"High (5+ roads):  {high:,}  ({high/len(degrees)*100:.1f}%)")

TASK 8: Intersection Categories



Category Summary:
Low  (1-2 roads): 37,720  (43.1%)
Medium (3-4 roads): 49,593  (56.6%)
High (5+ roads):  262  (0.3%)
